# Customer Churn Prediction and Retention Priority List

Telco Customer Churn dataset (7,043 customers). This notebook cleans the data, trains a logistic regression model, and ranks current customers by **expected monthly loss = churn probability x monthly charges**.

**Before running:** place `churn_cleaned5.csv` (the cleaned dataset) in the same folder as this notebook.

## 1. Load the data

In [1]:
import pandas as pd

path = "churn_cleaned5.csv"
df = pd.read_csv(path)

print("Before:", df.shape)
# keep only the 21 real columns (the CSV export contains 2 extra scratch columns)
df = df.iloc[:, :21]
print("After:", df.shape)
print(df.columns.tolist())
df.head()

Before: (7043, 23)
After: (7043, 21)
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Check data types and the target

In [2]:
df.info()
print(df["Churn"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## 3. Prepare features (X) and target (y)

`Churn` becomes 1 (Yes) or 0 (No). Text columns are one-hot encoded.

In [3]:
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
X = pd.get_dummies(X, drop_first=True, dtype=int)

print(X.shape)
print(round(y.mean(), 4))
X.head()

(7043, 30)
0.2654


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56.95,1889.50,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42.30,1840.75,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70.70,151.65,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0


## 4. Train and evaluate the model

80/20 stratified split, logistic regression with feature scaling.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print("Train rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("ROC AUC:", round(roc_auc_score(y_test, prob), 4))
print("Baseline accuracy (always predict no churn):", round(1 - y_test.mean(), 4))

Train rows: 5634 | Test rows: 1409
Accuracy: 0.807
ROC AUC: 0.8418
Baseline accuracy (always predict no churn): 0.7346


## 5. Confusion matrix and cut-off trade-off

In [5]:
from sklearn.metrics import confusion_matrix, classification_report, recall_score, precision_score

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["Stayed", "Churned"]))

for t in [0.5, 0.4, 0.3]:
    p = (prob >= t).astype(int)
    print("cut-off", t,
          "| recall:", round(recall_score(y_test, p), 2),
          "| precision:", round(precision_score(y_test, p), 2),
          "| flagged:", p.sum())

[[925 110]
 [162 212]]
              precision    recall  f1-score   support

      Stayed       0.85      0.89      0.87      1035
     Churned       0.66      0.57      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

cut-off 0.5 | recall: 0.57 | precision: 0.66 | flagged: 322
cut-off 0.4 | recall: 0.67 | precision: 0.57 | flagged: 441
cut-off 0.3 | recall: 0.75 | precision: 0.52 | flagged: 543


## 6. Priority list

Score every customer, then rank **current** customers (Churn = No) by expected monthly loss.

In [6]:
df["churn_prob"] = model.predict_proba(X)[:, 1]
df["expected_monthly_loss"] = (df["churn_prob"] * df["MonthlyCharges"]).round(2)

active = df[df["Churn"] == "No"]

cols = ["customerID", "Contract", "InternetService", "tenure",
        "MonthlyCharges", "churn_prob", "expected_monthly_loss"]
priority = active.sort_values("expected_monthly_loss", ascending=False)[cols]

out_path = "priority_list.csv"
priority.to_csv(out_path, index=False)

priority.head(10)

,customerID,Contract,InternetService,tenure,MonthlyCharges,churn_prob,expected_monthly_loss
3159,5150-ITWWB,Month-to-month,Fiber optic,3,94.85,0.846996,80.34
1568,3292-PBZEJ,Month-to-month,Fiber optic,11,111.40,0.702638,78.27
4701,5956-YHHRX,Month-to-month,Fiber optic,21,104.35,0.738542,77.07
4039,8161-QYMTT,Month-to-month,Fiber optic,7,94.10,0.813316,76.53
3863,0365-BZUWY,Month-to-month,Fiber optic,17,102.55,0.741428,76.03
5019,3320-VEOYC,Month-to-month,Fiber optic,14,95.60,0.793512,75.86
4618,6350-XFYGW,Month-to-month,Fiber optic,4,94.75,0.795378,75.36
2387,6734-GMPVK,Month-to-month,Fiber optic,5,105.30,0.713501,75.13
5072,5995-SNNEW,Month-to-month,Fiber optic,23,104.45,0.718452,75.04
3954,7225-CBZPL,Month-to-month,Fiber optic,17,94.80,0.787007,74.61


## 7. Main churn drivers

Positive coefficients push churn up, negative ones push it down. Treat the charge coefficients with caution because those columns overlap with the service columns.

In [7]:
coefs = model.named_steps["logisticregression"].coef_[0]
drivers = pd.Series(coefs, index=X.columns).sort_values()

print("Pushes churn DOWN (most negative):")
print(drivers.head(5).round(2))
print()
print("Pushes churn UP (most positive):")
print(drivers.tail(5).round(2))

Pushes churn DOWN (most negative):
tenure               -1.24
MonthlyCharges       -0.92
Contract_Two year    -0.59
Contract_One year    -0.29
OnlineSecurity_Yes   -0.12
dtype: float64

Pushes churn UP (most positive):
MultipleLines_Yes              0.22
StreamingTV_Yes                0.26
StreamingMovies_Yes            0.26
TotalCharges                   0.51
InternetService_Fiber optic    0.78
dtype: float64


## 8. Risk bands and sanity check

Note: these rates use the same data the model learned from, so they look slightly better than they would on brand-new customers.

In [8]:
df["risk_band"] = pd.cut(df["churn_prob"], bins=[0, 0.3, 0.6, 1.0],
                         labels=["Low", "Medium", "High"], include_lowest=True)

scored_path = "customers_scored.csv"
df.to_csv(scored_path, index=False)

print(df["risk_band"].value_counts())

check = df.groupby("risk_band", observed=True).agg(
    customers=("customerID", "count"),
    actually_churned=("Churn", lambda s: (s == "Yes").sum()),
    avg_monthly_charge=("MonthlyCharges", "mean")
)
check["actual_churn_rate_pct"] = (100 * check["actually_churned"] / check["customers"]).round(1)
check["avg_monthly_charge"] = check["avg_monthly_charge"].round(2)
check

risk_band
Low       4370
Medium    1631
High      1042
Name: count, dtype: int64


,customers,actually_churned,avg_monthly_charge,actual_churn_rate_pct
risk_band,,,,
Low,4370,429,57.28,9.8
Medium,1631,692,73.52,42.4
High,1042,748,82.42,71.8
